In [1]:
!pip install pyspark

In [92]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, count,avg, min, max

In [93]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Assignment") \
    .config("spark.hadoop.io.native.lib.available", "false") \
    .getOrCreate()
print(spark.version)

4.1.2


In [95]:
df_csv = (
    spark.read
    .option("header", True)
    .option("nullValue", "null")
    .option("inferSchema", True)
    .csv("Downloads/synthetic_dataset.csv")
)
df_parquet = spark.read.parquet("Downloads/synthetic_dataset_parquet")

In [98]:
df_csv.show()

+--------+------+------------------+------------+--------+
|Category| Price|            Rating|       Stock|Discount|
+--------+------+------------------+------------+--------+
|    NULL|5548.0|1.8703220155006277|        NULL|     0.0|
|    NULL|3045.0| 4.757798057831803|        NULL|    38.0|
|    NULL|4004.0|              NULL|    In Stock|     0.0|
|    NULL|4808.0| 1.492084885301209|        NULL|    33.0|
|    NULL|1817.0|              NULL|Out of Stock|    23.0|
|    NULL|3522.0|              NULL|        NULL|    NULL|
|       C| 667.0|3.6683407484908157|    In Stock|    41.0|
|       A|7125.0| 4.983998118378299|Out of Stock|     7.0|
|       A|2777.0| 2.678384421487482|    In Stock|     6.0|
|    NULL| 463.0| 4.626187280896929|        NULL|     3.0|
|       A|1151.0| 2.947838084766442|        NULL|    NULL|
|       A|3772.0| 4.890749963096654|    In Stock|    45.0|
|    NULL|7719.0|2.9822417824846874|    In Stock|     4.0|
|       C|8416.0|1.2709427320459685|        NULL|    29.

In [99]:
df_csv.filter(F.col("category") == "A").show()

+--------+------+------------------+------------+--------+
|Category| Price|            Rating|       Stock|Discount|
+--------+------+------------------+------------+--------+
|       A|7125.0| 4.983998118378299|Out of Stock|     7.0|
|       A|2777.0| 2.678384421487482|    In Stock|     6.0|
|       A|1151.0| 2.947838084766442|        NULL|    NULL|
|       A|3772.0| 4.890749963096654|    In Stock|    45.0|
|       A|7936.0| 3.032832041059039|    In Stock|    44.0|
|       A|2260.0|              NULL|    In Stock|    37.0|
|       A|6304.0|              NULL|        NULL|    31.0|
|       A|5006.0|3.8356362737695107|Out of Stock|    NULL|
|       A|9516.0|1.6573504366560314|        NULL|     5.0|
|       A|1962.0| 4.976201301561568|    In Stock|     6.0|
|       A|4077.0|              NULL|Out of Stock|    28.0|
|       A|4752.0| 4.589987920715608|    In Stock|     1.0|
|       A|6557.0|              NULL|Out of Stock|    32.0|
|       A|2466.0|              NULL|        NULL|    12.

In [100]:
df_csv.filter(F.col("Price") > 8500).show()


+--------+------+------------------+------------+--------+
|Category| Price|            Rating|       Stock|Discount|
+--------+------+------------------+------------+--------+
|       B|8530.0|              NULL|        NULL|    10.0|
|       B|9319.0|3.4790635118573126|    In Stock|    28.0|
|    NULL|9307.0|              NULL|Out of Stock|    21.0|
|    NULL|9230.0|              NULL|        NULL|     8.0|
|    NULL|8675.0| 2.778260648243727|Out of Stock|    NULL|
|       C|9870.0|2.8036864935783328|Out of Stock|    37.0|
|       B|9795.0|              NULL|    In Stock|    17.0|
|    NULL|9888.0|              NULL|        NULL|    29.0|
|    NULL|8773.0| 1.865998304807294|Out of Stock|    17.0|
|       A|9516.0|1.6573504366560314|        NULL|     5.0|
|    NULL|8631.0|3.3084805096007273|    In Stock|    10.0|
|       B|8723.0|3.7509731294173987|        NULL|    41.0|
|    NULL|8651.0| 4.282273713935922|        NULL|     2.0|
|       D|9493.0|2.7804614226232114|Out of Stock|    34.

In [101]:
df_transformed = (
    df_csv.withColumnRenamed("Price", "unit_price")
    .withColumnRenamed("Stock", "Stock Available"))

In [102]:
df_transformed.columns

['Category', 'unit_price', 'Rating', 'Stock Available', 'Discount']

In [103]:
lazy_chain = (
    df_transformed
    .filter(F.col("category").isin(["A", "B"]))
    .groupBy("category")
    .agg(F.sum("Rating").alias("Total Rating"))
)
print("Rating by category (A & B):")
lazy_chain.show()

Rating by category (A & B):
+--------+-----------------+
|category|     Total Rating|
+--------+-----------------+
|       B|635.2077968350562|
|       A|642.6588619233986|
+--------+-----------------+



In [104]:
df_transformed.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_transformed.columns
]).show()

+--------+----------+------+---------------+--------+
|Category|unit_price|Rating|Stock Available|Discount|
+--------+----------+------+---------------+--------+
|    2748|       174|  2050|           1352|     392|
+--------+----------+------+---------------+--------+



In [108]:
df_transformed=df_transformed.filter(F.col("Stock Available").isNotNull())

In [109]:
df_transformed.count()

3010

In [110]:
mean_discount=df_transformed.select(F.mean("Discount")).first()[0]
df_transformed = df_transformed.fillna({"Discount": mean_discount})

mean_Rating=df_transformed.select(F.mean("Rating")).first()[0]
df_transformed = df_transformed.fillna({"Rating": mean_Rating})

mean_unit_price=df_transformed.select(F.mean("unit_price")).first()[0]
df_transformed = df_transformed.fillna({"unit_price": mean_unit_price})

In [111]:
mode_category = df_transformed.groupBy("Category") \
    .count() \
    .orderBy(F.asc("count")) \
    .first()[0]

df_transformed = df_transformed.fillna({"Category": mode_category})

In [91]:
df_transformed.write.mode("overwrite").option("header", True).csv("Downloads/output_6.csv")